
## Training a U-Net to count cells

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AIM-Net/SPAOM-CW20-DeepTrack/blob/main/UNet-train.ipynb)

[Repository](https://github.com/AIM-Net/SPAOM-CW20-DeepTrack)

SPAOM 2026<br>
6-9 Oct 2026<br>
Guillem Guigó & Carlo Manzo (UVic-UCC)

*This community workshop is part of the activities of AIM-Net (RED2024-153844-T, funded by MICIU/AEI/10.13039/501100011033).*

**Companion notebook to [`CW20_Guigo_SPAOM2026.ipynb`](CW20_Guigo_SPAOM2026.ipynb)**, which builds a simulation of fluorescent cell nuclei and stops once the pipeline is ready. This one picks it up and puts it to work: it trains a U-Net **only on simulated images** and then counts cells in the real [BBBC039](https://bbbc.broadinstitute.org/BBBC039) images.

The notebook is self-contained, so you can run it on its own. Sections 0 and 1 reproduce the simulation from the main notebook; the new material starts at section 2.

The code is amply based on Code Example 5-B from the book:

<div style="background-color: #f0f8ff; border: 2px solid #4682b4; padding: 10px;">
**Deep Learning Crash Course**  
Giovanni Volpe, Benjamin Midtvedt, Jesús Pineda, Henrik Klein Moberg, Harshith Bachimanchi, Joana B. Pereira, Carlo Manzo  
No Starch Press, San Francisco (CA), 2026  
ISBN-13: 9781718503922  

[https://nostarch.com/deep-learning-crash-course](https://nostarch.com/deep-learning-crash-course)

You can find the other notebooks on the [Deep Learning Crash Course GitHub page](https://github.com/DeepTrackAI/DeepLearningCrashCourse).
</div>


*This notebook takes a while to run.* Generating the simulated dataset is the slow part, and training adds a few minutes on a laptop CPU. See the note in section 2.2 if you want to make it faster.



### 0. Setup

If you are running on **Google Colab / Kaggle**, uncomment and run the next cell. Locally, install the packages once in your environment (`pip install -e .` from the repository root).


In [ ]:
# !pip install deeptrack deeplay

In [ ]:

import os

import deeplay as dl
import deeptrack as dt
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from skimage import morphology as skmorph

np.random.seed(2026)  # Makes a top-to-bottom run reproducible.



### 1. The simulated sample

A condensed version of section 2 of the main notebook. We first load the real images we want to imitate, then rebuild the simulation that produces look-alikes of them.


In [ ]:
if not os.path.exists("cell_counting_dataset"):
    os.system("git clone https://github.com/DeepTrackAI/cell_counting_dataset")

root = os.path.join("cell_counting_dataset", "cell nuclei")
image_dir = os.path.join(root, "images")
mask_dir = os.path.join(root, "masks")

image_files = sorted(os.listdir(image_dir))
mask_files = sorted(os.listdir(mask_dir))

In [ ]:
def load_pair(idx):
    """Loads an image with ground truth from file."""
    image = Image.open(os.path.join(image_dir, image_files[idx]))
    image = np.array(image, dtype=np.float32)
    image = image/image.max()
    image = torch.from_numpy(image)

    mask = Image.open(os.path.join(mask_dir, mask_files[idx])).convert("RGB")
    mask = np.array(mask, np.uint8)
    mask = mask.max(axis=2)
    mask = (mask > 0).astype(np.float32)
    mask = torch.from_numpy(mask)

    return image, mask

image, mask = load_pair(0)
print(image.shape, mask.shape)

# Plot a sample.
fig, axs = plt.subplots(1, 2, figsize=(10, 5))
axs[0].imshow(image, cmap="gray")
axs[1].imshow(mask, cmap="gray")
plt.tight_layout()
plt.show()


#### 1.1 Cell-like scatterers

To a first approximation a nucleus is an ellipsoid, so we draw its area, elongation, orientation and brightness at random. The `NA` of the microscope is randomized too, since focus and optical quality vary from one experimental image to the next.


In [ ]:
IMAGE_SIZE = 256

# Helper function to keep axis within reasonable parameters.
def random_ellipse_axes():
    """Return the three axes of an ellipse."""
    ellipse_area = (np.random.uniform(3, 4)) ** 2
    radius_ratio = np.random.uniform(1, 1.5)
    major_axis = np.sqrt(ellipse_area) * radius_ratio
    minor_axis = np.sqrt(ellipse_area) / radius_ratio
    z_axis = np.sqrt(ellipse_area) * np.random.uniform(0.2, 0.4)
    return (major_axis, minor_axis, z_axis)* dt.units.um

# Define the ellipse.
ellipse = dt.Ellipsoid(
    radius = random_ellipse_axes,
    intensity=lambda: np.random.uniform(0.5, 1.5),
    position=lambda: np.random.uniform(5, IMAGE_SIZE - 5, size=2),
    rotation=lambda: np.random.uniform(0, 2 * np.pi),
)

# Define the microscope.
optics = dt.Fluorescence(
    resolution=1e-6,
    magnification=6,
    wavelength=400e-9,
    NA=lambda: np.random.uniform(0.9, 1.1),
    output_region=(0, 0, IMAGE_SIZE, IMAGE_SIZE),
)

# Image the ellipses with a fluorescence microscope.
sim_im_pip = optics(ellipse)

# Plot some samples.
fig, axs = plt.subplots(1, 5, figsize=(10, 2))
for i, ax in enumerate(axs):
    sim_im_pip.update()
    image = sim_im_pip()
    ax.imshow(image, cmap="gray")
    if i != 0: ax.axis("off")
plt.tight_layout()
plt.show()


#### 1.2 Realistic cells

Plain ellipsoids are too smooth and too uniform to pass for real nuclei. An **elastic transformation** bends the outline into something irregular, and **texture** comes from multiplying the nuclei with Poisson noise blurred at three different length scales. We then place them so that they do not overlap, image them, and match the background level and intensity range of the real data.


In [ ]:
ellipse = dt.Ellipsoid(
    radius = random_ellipse_axes,
    intensity=lambda: np.random.uniform(0.5, 1.5),
    position=lambda: np.random.uniform(5, IMAGE_SIZE - 5, size=2),
    rotation=lambda: np.random.uniform(0, 2 * np.pi),
)

# Add elastic transformations.
synthetic_nuclei = (
    (ellipse ^ (lambda: np.random.randint(5, 10)))
    >> dt.Pad(px=(10, 10, 10, 10), keep_size=False)
    >> dt.ElasticTransformation(alpha=100, sigma=10, order=1)
    >> dt.CropTight()
)

# Define Poisson noise and Gaussian blur.
long_range_noise = (synthetic_nuclei >> dt.Poisson(snr=0.2)
                    >> dt.GaussianBlur(sigma=3.5))
short_range_noise = (synthetic_nuclei >> dt.Poisson(snr=1.0)
                     >> dt.GaussianBlur(sigma=1.5))
random_range_noise = (
    synthetic_nuclei
    >> dt.Poisson(snr=lambda: np.random.uniform(0.5, 1.5))
    >> dt.GaussianBlur(sigma=lambda: np.random.uniform(0.75, 1.5))
)

# Combine the augmentations.
noisy_synthetic_nuclei = (
    synthetic_nuclei
    * (long_range_noise + short_range_noise + random_range_noise) / 3
)

# Ensure non-overlapping cells.
non_overlap_nuclei = dt.NonOverlapping(noisy_synthetic_nuclei, min_distance=2)

# Define the microscope.
optics = dt.Fluorescence(
    resolution=1e-6, magnification=6, wavelength=400e-9,
    NA=lambda: np.random.uniform(0.9, 1.1),
    output_region=(0, 0, IMAGE_SIZE, IMAGE_SIZE),
)

# Define simulation pipeline with some background noise.
sim_im_pip = (optics(non_overlap_nuclei)
              >> dt.Gaussian(sigma=lambda: np.random.uniform(0, 0.1))
              >> dt.Divide(lambda: np.random.uniform(14, 20))
              >> dt.Add(lambda: np.random.uniform(-0.05, 0.15))
              >> dt.Clip(0, 1) >> dt.AsType("float"))

# Plot some samples.
fig, axs = plt.subplots(1, 5, figsize=(10, 2))
for i, ax in enumerate(axs):
    sim_im_pip.update()
    image = sim_im_pip()
    ax.imshow(image, cmap="gray")

plt.tight_layout()
plt.show()


### 2. From simulation to training data

Everything so far produces images. To train a supervised model we also need the labels that go with them, and a dataset object that PyTorch understands.


#### 2.1 Ground-truth masks

The labels come for free: `dt.SampleToMasks` builds a binary mask from the very same scatterers that produced the image. We erode each nucleus slightly so that neighbors stay separated, which is what will allow us to count them as individual objects:

In [ ]:
def get_mask(radius):
    """Apply isotropic erosion to a binary mask."""
    def inner(mask):
        mask = np.sum(mask, -1, keepdims=True) > 0
        mask = np.pad(mask, [(1, 1), (1, 1), (0, 0)], mode="constant")
        mask = skmorph.isotropic_erosion(mask, radius=radius)
        return mask[1:-1, 1:-1]
    return inner

# Define mask pipeline.
sim_mask_pip = (
    noisy_synthetic_nuclei
    >> dt.SampleToMasks(get_mask, radius=2, output_region=optics.output_region,
                       merge_method="or")
    >> dt.AsType("float")
)

#### 2.2 The training dataset

The `&` operator pairs every image with its mask, and `dt.pytorch.Dataset` exposes the result as a standard PyTorch dataset. Samples are generated on demand and cached; `replace=0.01` regenerates a small fraction of them at each epoch, so the network keeps seeing fresh images:

In [ ]:
# Combine image simulation and mask pipeline.
sim_im_mask_pip = ((sim_im_pip & sim_mask_pip) >> dt.MoveAxis(2, 0)
                   >> dt.pytorch.ToTensor(dtype=torch.float))

# Generate the training dataset. Samples are produced on demand, and
# dt.NonOverlapping makes each one expensive (~10 s on a laptop CPU), so the
# first epoch is slow. Lower `length` to shorten it.
train_dataset = dt.pytorch.Dataset(sim_im_mask_pip, length=512, replace=0.01)

Let's sample the dataset and compare it with the real images above. The intensity histogram is a quick way to check that the simulation lands in the same range as the experimental data:

In [ ]:
sim_im, sim_mask = train_dataset[np.random.choice(len(train_dataset))]

fig, axs = plt.subplots(1, 3, figsize=(12, 4))

# Plot image.
axs[0].imshow(sim_im.permute(1, 2, 0), vmin=0, vmax=1, cmap="gray")
axs[0].set_title("Simulated Image", fontsize=20)

# Plot mask.
axs[1].imshow(sim_mask.permute(1, 2, 0), vmin=0, vmax=1, cmap="gray")
axs[1].set_title("Simulated Mask", fontsize=20)
axs[1].axis("off")

# plot intensity distribution.
axs[2].hist(np.array(sim_im).flatten(), bins=200, range=(0, 1))
axs[2].set_xlabel("Pixel Intensity", fontsize=16)
axs[2].set_xlim([0, 1])
axs[2].set_ylabel("Log Frequency", fontsize=16)
axs[2].set_yscale("log")
axs[2].set_title("Pixel Intensity Distribution", fontsize=20)

plt.tight_layout()
plt.show()

### 3. Training a U-Net

A **U-Net** is a convolutional network that returns an image of the same size as its input, which makes it a natural choice to turn a microscopy image into a segmentation mask. Counting is then just a matter of labelling the connected components of that mask. The network is trained exclusively on the images we simulated in section 1.

#### 3.1 Defining the U-Net

We build the network with [deeplay](https://github.com/DeepTrackAI/deeplay), the deep-learning companion library of DeepTrack2. The three channel sizes set how far the image is downsampled and then upsampled again. Since the network has to decide, for every pixel, whether it belongs to a nucleus or to the background, we train it with a binary cross-entropy loss:

In [ ]:
# Using deeplay U-Net template. 
model = dl.UNet2d(
    in_channels = 1,
    channels=[16, 32, 64,],
    out_channels=1,
)

# Compile as a regressor.
unet_regressor = dl.Regressor(
    model=model, loss=torch.nn.BCEWithLogitsLoss(), optimizer=dl.Adam(),
).create()

print(unet_regressor)

#### 3.2 Training the U-Net

Each epoch draws batches from the simulated dataset, so the network never sees a single real image during training.

This may take a few minutes on a laptop CPU, and much less on a GPU.

In [ ]:
# Define training data loader.
train_loader = dl.DataLoader(train_dataset, batch_size=8, shuffle=True)

# Define model trainer.
unet_trainer = dl.Trainer(max_epochs=50, accelerator="auto")

# Start training.
unet_trainer.fit(unet_regressor, train_loader) 
unet_trainer.history.plot()
unet_regressor.eval();

#### 3.3 Evaluating on real images

This is the moment of truth: we apply the network, trained **only on simulated data**, to the real BBBC039 images. For each one we threshold the predicted mask, label its connected components to obtain a cell count, and compare with the count from the annotated mask:

In [ ]:
images, masks, preds = [], [], []
with torch.no_grad():
    for idx in range(len(image_files)):
        image, mask = load_pair(idx)

        image = image.unsqueeze(0).unsqueeze(0)
        mask  = mask.unsqueeze(0).unsqueeze(0)

        pred = unet_regressor(image)
        pred = torch.sigmoid(pred).cpu()

        images.append(image.cpu())
        masks.append(mask.cpu())
        preds.append(pred)

images = torch.cat(images, dim=0)
masks  = torch.cat(masks, dim=0)
preds  = torch.cat(preds, dim=0)

true_count = np.array([skmorph.label(l.squeeze()).max() for l in masks])
pred_count = np.array([skmorph.label(p.squeeze() > .995).max() for p in preds])

mae = abs(pred_count - true_count).mean()
mpe = (abs(pred_count[true_count > 0] - true_count[true_count > 0])
       / (true_count[true_count > 0])).mean()

print(f"Mean absolute error: {mae:.2f} cells ({mpe:.1%})")

Let's look at one image, its annotation and the prediction side by side:

In [ ]:
random_sample = np.random.choice(len(images))

fig, axs = plt.subplots(1, 3, figsize=(10, 4))

axs[0].imshow(images[random_sample].squeeze(), vmin=0, vmax=1, cmap="gray")
axs[0].set_title("Image", fontsize=16)

axs[1].imshow(masks[random_sample].squeeze(), vmin=0, vmax=1, cmap="gray")
axs[1].set_title(f"Label - {true_count[random_sample]} cells", fontsize=16)
axs[1].axis("off")

axs[2].imshow(preds[random_sample].squeeze(), cmap="gray")
axs[2].set_title(f"Prediction - {pred_count[random_sample]} cells", fontsize=16)
axs[2].axis("off")

plt.tight_layout()
plt.show()

### Exercises

**Exercise 3.** Plot the predicted cell counts against the true counts for the whole dataset. Is the error uniform, or does the network struggle more on the crowded images?

In [ ]:
# Your code here.


**Exercise 4.** Pick one aspect of the simulation and change it in 1.2: the number of nuclei per image, the amount of texture and noise, the range of sizes in `random_ellipse_axes`, or the background level. Regenerate the dataset, retrain the U-Net and compare the error on the real images with the one you obtained above. Which change hurts the most?

In [ ]:
# Your code here.
